Download daily financial data set for 50-100 stocks covering a time period of 10 years
For ech day, store the date(yyyymmdd), the adjusted close price (last price of the day) and the volume (num of shares traded)
Can use Python library yfinance

In [3]:
!pip install yfinance

  Using cached yfinance-1.2.0-py2.py3-none-any.whl.metadata (6.1 kB)
  Using cached multitasking-0.0.12.tar.gz (19 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached pytz-2026.1.post1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached frozendict-2.4.7-py3-none-any.whl.metadata (23 kB)
  Using cached peewee-4.0.1-py3-none-any.whl.metadata (8.5 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached curl_cffi-0.13.0-cp39-abi3-win_amd64.whl.metadata (13 kB)
  Using cached protobuf-7.34.0-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached soupsieve-2.8.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)

In [3]:
!pip install lxml

   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 4.0/4.0 MB 80.3 MB/s  0:00:00


In [1]:
import yfinance as yf
from datetime import datetime
import pandas as pd
import requests
from io import StringIO

In [2]:
#Time period to fetch from
start_data = datetime(year=2015, month=1, day=1)


end_data = datetime(year=2026, month = 3, day =9)
#list of stocks
wikipedia_page = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

response = requests.get(wikipedia_page, headers=headers)
response.raise_for_status()

list_of_stocks = pd.read_html(StringIO(response.text))[0]

ticker_symbols = list_of_stocks['Symbol'].tolist()
ticker_symbols_75 = ticker_symbols[:75]


In [3]:
all_data = []
ticker_symbols_75_clean = [t.replace('.', '-') for t in ticker_symbols_75]

data = yf.download(tickers=ticker_symbols_75_clean, start=start_data, end=end_data, group_by='ticker')

#in data: key: [[date, close, volume], [date, close, volume], ...]]
for ticker in ticker_symbols_75_clean:
    hist_data = data[ticker][['Close', 'Volume']].copy()
    hist_data["Ticker"] = ticker
    hist_data.reset_index(inplace=True) #Make date column
    all_data.append(hist_data)
    
    
combined_data = pd.concat(all_data, ignore_index=True)
print(combined_data.head())

[*********************100%***********************]  75 of 75 completed


Price       Date      Close     Volume Ticker
0     2015-01-02  95.679237  2531214.0    MMM
1     2015-01-05  93.521400  4416708.0    MMM
2     2015-01-06  92.524139  4224272.0    MMM
3     2015-01-07  93.194824  3685235.0    MMM
4     2015-01-08  95.428459  3758908.0    MMM


Test set: all data between 20250101 and now
Val set: all data between 20240101 and 20241231
Training data: 2015 to 20231231

In [4]:
train_data = combined_data[combined_data["Date"] <= datetime(year=2023, month=12, day=31)]
validation_data = combined_data[(combined_data["Date"] > datetime(year=2023, month=12, day=31)) & (combined_data["Date"] <= datetime(year=2024, month=12, day=31))]
test_data = combined_data[combined_data["Date"] > datetime(year=2024, month=12, day=31)]

#X_train, y_train = train_data[['Date', 'Ticker', "Volume"]], train_data['Close']
#X_val, y_val = validation_data[['Date', 'Ticker', "Volume"]], validation_data['Close']
#X_test, y_test = test_data[['Date', 'Ticker', "Volume"]], test_data['Close']



Add another column for each time series and in each set
Column; int(1) if closing stock price on the next day is >=3% higher than price on current day, 0 otherwise


In [5]:

#Takes like 0 sec
def add_increase(df):
    df["Increase"] = df.groupby("Ticker")["Close"].shift(-1) >= 1.03 * df['Close'] #Groupby means it looks at the data for each tiocker seperately, the shift means that we look at the data for next step
    df["Increase"] = df["Increase"].astype(int) #Convert boolean to int (True becomes 1, False becomes 0)
    return df

train_data = add_increase(train_data)
train_data = train_data.dropna()
validation_data = add_increase(validation_data)
validation_data = validation_data.dropna()
test_data = add_increase(test_data)
test_data = test_data.dropna()


""" #takes 4 min
for ticker in ticker_symbols_75:
    data_for_ticker = train_data[train_data['Ticker'] == ticker].reset_index() #Reset index makes hte index into a regular column, so we can refer to the different dates as 0,1,2, etc. instead of like the date
    
    for date_indx in range(len(data_for_ticker)-1):
        closing_today = data_for_ticker.loc[date_indx, 'Close']
        closing_tomorrow = data_for_ticker.loc[date_indx + 1, 'Close']
        
        date_today = data_for_ticker.loc[date_indx, "Date"]
        
        if closing_tomorrow >= 1.03 * closing_today:
            train_data.loc[(train_data["Ticker"] == ticker) & (train_data["Date"] == date_today)]["Increase"] = 1
        else:
            #print(train_data[ticker])
            train_data.loc[(train_data["Ticker"] == ticker) & (train_data["Date"] == date_today)]["Increase"] = 0

"""
print(train_data)


#iloc uses integers, loc uses labels

            

Price        Date       Close     Volume Ticker  Increase
0      2015-01-02   95.679237  2531214.0    MMM         0
1      2015-01-05   93.521400  4416708.0    MMM         0
2      2015-01-06   92.524139  4224272.0    MMM         0
3      2015-01-07   93.194824  3685235.0    MMM         0
4      2015-01-08   95.428459  3758908.0    MMM         0
...           ...         ...        ...    ...       ...
210199 2023-12-22  193.450317   568800.0     BR         0
210200 2023-12-26  195.727127   981100.0     BR         0
210201 2023-12-27  197.907043   537500.0     BR         0
210202 2023-12-28  199.418457   535500.0     BR         0
210203 2023-12-29  199.340912   441100.0     BR         0

[166500 rows x 5 columns]


Train LSTM network 
Output: 0 or 1 (increase)
Input: Price and Volume 
For any stock: use first 10 days as warmup period  (ie do not use output). 
Compute error as difference between desired and actual output
Combine list of errors for each stock to total list of errors
This list should be used (after normalisation) in the loss functiopn when trainin the LSTM network
Use holdout validation


In [22]:
!pip install tensorflow

   ---------------------------------------- 0.0/350.8 MB ? eta -:--:--
   -- ------------------------------------ 21.8/350.8 MB 105.9 MB/s eta 0:00:04
   ----- --------------------------------- 45.9/350.8 MB 112.3 MB/s eta 0:00:03
   ------- ------------------------------- 70.3/350.8 MB 114.9 MB/s eta 0:00:03
   ---------- ---------------------------- 94.4/350.8 MB 115.9 MB/s eta 0:00:03
   ------------- ------------------------ 120.1/350.8 MB 116.2 MB/s eta 0:00:02
   --------------- ---------------------- 143.9/350.8 MB 116.4 MB/s eta 0:00:02
   ----------------- -------------------- 157.0/350.8 MB 109.1 MB/s eta 0:00:02
   ------------------- ------------------ 181.4/350.8 MB 110.5 MB/s eta 0:00:02
   ---------------------- --------------- 203.9/350.8 MB 110.5 MB/s eta 0:00:02
   ------------------------ ------------- 227.5/350.8 MB 110.2 MB/s eta 0:00:02
   --------------------------- ---------- 251.7/350.8 MB 111.0 MB/s eta 0:00:01
   ----------------------------- -------- 275.5/3

In [6]:
import tensorflow.keras.models as models
import tensorflow.keras.layers as layers
import tensorflow.keras.callbacks as callbacks
import numpy as np

from sklearn.preprocessing import MinMaxScaler


window_size = 20


In [ ]:
import torch
if torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")
  
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [64]:
def create_sequence(data, seq_length):
    X, y = [], []
    for i in range(0, len(data) -seq_length):
        interval = data.iloc[i:i+seq_length]
        
        seq = np.column_stack((interval["Close"].values, interval["Volume"].values))
        X.append(seq)
        step = data.iloc[i+seq_length] #Wont that just predict a point whose volume and close value is not taken into account?
        y.append(step["Increase"])
        
        if i%1100 == 0:
            print(f"Processed {i} out of {len(data)} rows")

    return np.array(X), np.array(y)
seq_length=32
X_train, y_train = create_sequence(train_data, seq_length)
X_val, y_val = create_sequence(validation_data, seq_length)
X_test, y_test = create_sequence(test_data, seq_length)

Processed 0 out of 166500 rows
Processed 1100 out of 166500 rows
Processed 2200 out of 166500 rows
Processed 3300 out of 166500 rows
Processed 4400 out of 166500 rows
Processed 5500 out of 166500 rows
Processed 6600 out of 166500 rows
Processed 7700 out of 166500 rows
Processed 8800 out of 166500 rows
Processed 9900 out of 166500 rows
Processed 11000 out of 166500 rows
Processed 12100 out of 166500 rows
Processed 13200 out of 166500 rows
Processed 14300 out of 166500 rows
Processed 15400 out of 166500 rows
Processed 16500 out of 166500 rows
Processed 17600 out of 166500 rows
Processed 18700 out of 166500 rows
Processed 19800 out of 166500 rows
Processed 20900 out of 166500 rows
Processed 22000 out of 166500 rows
Processed 23100 out of 166500 rows
Processed 24200 out of 166500 rows
Processed 25300 out of 166500 rows
Processed 26400 out of 166500 rows
Processed 27500 out of 166500 rows
Processed 28600 out of 166500 rows
Processed 29700 out of 166500 rows
Processed 30800 out of 166500 row

In [52]:
import tensorflow as tf
import os
import torch.nn as nn
import numpy as np

In [77]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout, output_size):
        super(LSTMModel, self).__init__()
        #self.embed = nn.Embedding(input_size) #converts word IDs to vectors. vocab_size = num of distinct tokens. Embed_size = how big each word vector is 
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.linear = nn.Linear(hidden_size, output_size)
    def forward(self, x, h): #x:(batch_size, seq_length), h: (h0, c0), h0: zeros in shape (num_layers, batch_size, hidden_size), c0: same
        #x = self.embed(x)
        out, (h,c) = self.lstm(x, h)
        out = out[:, -1, :]
        out = self.linear(out)
        return out, (h, c)

embed_size = 1
hidden_size = 32
num_layers = 2
learning_rate = 0.001
input_size = 2
dropout = 0.2


model = LSTMModel(input_size, hidden_size, num_layers, dropout, embed_size).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [84]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)

X_valid_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
y_valid_tensor = torch.tensor(y_val, dtype=torch.float32).to(device)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).to(device)

batch_size = 32

def training_epoch():
    model.train()
    
    for batch_idx in range(0, len(X_train_tensor), batch_size):
        X_batch = X_train_tensor[batch_idx:batch_idx+batch_size]
        y_batch = y_train_tensor[batch_idx:batch_idx+batch_size]
        
        h0 = torch.zeros(num_layers, X_batch.size(0), hidden_size, device = device)
        c0 = torch.zeros(num_layers, X_batch.size(0), hidden_size, device = device)
        
        #X_batch = X_batch.view(32,32,2) 
        #print("H")
        #print(X_batch.shape)
        pred, _ = model(X_batch, (h0, c0))
        
        pred = pred.reshape(-1)
        
        
        loss = criterion(pred, y_batch)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

def evaluate():
    model.eval()
    
    with torch.no_grad():
        error = []
        for val_batch_idx in range(0, len(X_valid_tensor), batch_size):
            X_val_batch = X_valid_tensor[val_batch_idx: val_batch_idx + batch_size]
            y_val_batch = y_valid_tensor[val_batch_idx: val_batch_idx + batch_size]
            
            h0_val = torch.zeros(num_layers, X_val_batch.size(0), hidden_size, device = device)
            c0_val = torch.zeros(num_layers, X_val_batch.size(0), hidden_size, device = device)
            
            val_pred, _ = model(X_val_batch, (h0_val, c0_val))        
            
            error.append(torch.abs(y_val_batch - val_pred).mean().item())
            
        avg_error = np.mean(error)
        
        return avg_error
        
    
num_epochs = 10

for epoch in range(num_epochs):
    training_epoch()
    error = evaluate()
    print(f"Epoch {epoch+1}/{num_epochs}, Validation Error: {error}")

Epoch 1/10, Validation Error: 3.2718453471943483
Epoch 2/10, Validation Error: 3.280161865686966
Epoch 3/10, Validation Error: 3.297653914306123
Epoch 4/10, Validation Error: 3.3011961047932252
Epoch 5/10, Validation Error: 3.2972682722544264
Epoch 6/10, Validation Error: 3.300690301798158
Epoch 7/10, Validation Error: 3.306427613355346
Epoch 8/10, Validation Error: 3.3137159355616164
Epoch 9/10, Validation Error: 3.3043610241453525
Epoch 10/10, Validation Error: 3.2969803890939486


In [ ]:
scaler = MinMaxScaler()

X_train_close, X_train_volume, y_train = train_data['Close'].values, train_data["Volume"].values, train_data['Increase'].values
#X_train_close = X_train_close.reshape(-1, 1) #Reshape to be 2D for scaler
#X_train_volume = X_train_volume.reshape(-1, 1) #Reshape to be 2D for scaler
print(X_train_close.shape)
print(X_train_volume.shape)

X_train = torch.stack((torch.tensor(X_train_close, dtype=torch.float32), torch.tensor(X_train_volume, dtype=torch.float32)), dim=0).to(device)

#X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1])) #Reshape to be compatible with LSTM input

X_val_close, X_val_volume, y_val = validation_data['Close'].values, validation_data["Volume"].values, validation_data['Increase'].values

X_val = torch.stack((torch.tensor(X_val_close, dtype=torch.float32), torch.tensor(X_val_volume, dtype=torch.float32)), dim=0).to(device)
#X_val = X_val.reshape((X_val.shape[0], 1, X_val.shape[1])) #Reshape to be compatible with LSTM input
X_test_close, X_test_volume, y_test = test_data['Close'].values, test_data["Volume"].values, test_data['Increase'].values

X_test = torch.stack((torch.tensor(X_test_close, dtype=torch.float32), torch.tensor(X_test_volume, dtype=torch.float32)), dim=0).to(device)
#X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1])) #Reshape to be compatible with LSTM input

print(X_train[0])

(166500,)
(166500,)
tensor(95.6792, device='cuda:0')


In [8]:
import tensorflow as tf
import os
import torch.nn as nn
import numpy as np

In [37]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(LSTMModel, self).__init__()
        self.layer_dim = layer_dim
        self.lstm = nn.LSTM(input_dim, layer_dim, batch_first=True, num_layers=num_layers, dropout=dropout)
        self.linear = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x, h): #x:(batch_size, seq_length), h: (h0, c0), h0: zeros in shape (layer_dim, batch_size, hidden_size), c0: same
        #x = self.embed(x)
        out, (h,c) = self.lstm(x, h)
        #out = out.reshape(out.size(0)*out.size(1), out.size(2))
        
        last_hidden = out[:, -1, :]
        
        out = self.linear(last_hidden)
        return out, (h, c)

input_dim = 2
hidden_dim = 64
num_layers = 2
dropout = 0.2
output_dim = 1
learning_rate = 0.01

model = LSTMModel(input_dim, hidden_dim, num_layers, output_dim).to(device)
criterion = nn.CrossEntropyLoss() #Add my own loss?!
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [38]:
num_epochs =10
print(X_train.shape)


seq_length = 32


model.train() 
for epoch in range(num_epochs):
    
    total_loss = 0
    #num_bath = 0
    error = []
    for batch_idx in range(0, len(X_train), seq_length):
        X_batch = X_train[batch_idx:batch_idx+seq_length]
        y_batch = y_train[batch_idx:batch_idx+seq_length]

        #h0 = torch.zeros(num_layers, X_batch.size(0), hidden_size, device=device)
        #c0 = torch.zeros(num_layers, y_batch.size(0), hidden_size, device=device)
        
        
        #print(X_batch.shape)

        pred, _ = model(X_batch)
        #pred_reshaped = pred.reshape(X_batch.size(0), X_batch.size(1), -1)
        #pred_last = pred_reshaped[:, -1, :]
        #y_flattened = y_batch.view(-1)
        loss = criterion(pred, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        #num_bath += 1
    
    model.eval()
    val_total_loss = 0
    val_num_batchs = 0
    with torch.no_grad():
        for val_batch_idx in range(0, len(X_val), seq_length):
            x_val_batch = X_val[val_batch_idx:val_batch_idx+seq_length]
            y_val_batch = y_val[val_batch_idx: val_batch_idx+ seq_length]
        
            #h0_val = torch.zeros(num_layers, x_val_batch.size(0), hidden_size, device=device)
            #c0_val = torch.zeros(num_layers, x_val_batch.size(0), hidden_size, device=device)

            predicted_val, _ = model(x_val_batch)

            #predicted_val_reshaped = predicted_val.reshape(x_val_batch.size(0), x_val_batch.size(1), -1)
#y_valid_flat = predicted_val.reshape(x_val_batch.size(0), x_val_batch.size(1), -1)
            #pred_last_val = predicted_val_reshaped[:, -1, :]

            #y_valid_flat = y_val_batch.view(-1)
            
            error_loc = np.abs(predicted_val - y_val_batch)
            error.extend(error_loc.flatten())
        
            #_, pred_val_val = torch.max(predicted_val, 1)
            #val_loss = criterion(pred_last_val, y_valid_flat)
            #val_total_loss += val_loss.item()
            #val_num_batchs += 1
    #avg_train_loss = total_loss / num_bath
    #avg_val_loss = val_total_loss / val_num_bath
    #val_perplexity = torch.exp(torch.tensor(avg_val_loss)).item()
    max = np.max(error)
    min = np.min(error)
    region = max - min
    error = (error - min) / region
    mean_error = np.mean(error)
    print(f"Epoch {epoch}, mean error = {mean_error:.4f}")

torch.Size([333000])


TypeError: LSTMModel.forward() missing 1 required positional argument: 'h'

In [66]:
import tensorflow as tf

In [72]:
#Version 1

lstm = models.Sequential([
    layers.LSTM(32, activation = 'tanh', input_shape=(X_train.shape[1], X_train.shape[2])),
    layers.Dense(1)
])

def custom_loss(y_true, y_pred):
    warmup = 10
    error = tf.abs(y_true[warmup:]-y_pred[warmup:])
    
    return tf.reduce_mean(error)
    
lstm.compile(optimizer='adam', loss=custom_loss, metrics=['mse'])

holdout = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
historical_data = lstm.fit(X_train, y_train, epochs = 50, batch_size=32, validation_data=(X_val, y_val), callbacks=[holdout])
predicted_values = lstm.predict(X_val) #Is it wrong to use the validation data in the last line if its also to be used here ?


Epoch 1/50


c:\Users\erkfeldt\AppData\Local\miniconda3\envs\pytorch-gpu\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


5204/5204 ━━━━━━━━━━━━━━━━━━━━ 5s 897us/step - loss: nan - mse: 0.0460 - val_loss: 0.0359 - val_mse: 0.0360
Epoch 2/50
5204/5204 ━━━━━━━━━━━━━━━━━━━━ 5s 861us/step - loss: nan - mse: 0.0460 - val_loss: 0.0362 - val_mse: 0.0360
Epoch 3/50
5204/5204 ━━━━━━━━━━━━━━━━━━━━ 5s 881us/step - loss: nan - mse: 0.0460 - val_loss: 0.0361 - val_mse: 0.0360
Epoch 4/50
5204/5204 ━━━━━━━━━━━━━━━━━━━━ 5s 904us/step - loss: nan - mse: 0.0460 - val_loss: 0.0358 - val_mse: 0.0360
Epoch 5/50
5204/5204 ━━━━━━━━━━━━━━━━━━━━ 5s 904us/step - loss: nan - mse: 0.0460 - val_loss: 0.0357 - val_mse: 0.0360
Epoch 6/50
5204/5204 ━━━━━━━━━━━━━━━━━━━━ 5s 882us/step - loss: nan - mse: 0.0460 - val_loss: 0.0357 - val_mse: 0.0360
Epoch 7/50
5204/5204 ━━━━━━━━━━━━━━━━━━━━ 5s 925us/step - loss: nan - mse: 0.0460 - val_loss: 0.0358 - val_mse: 0.0360
Epoch 8/50
5204/5204 ━━━━━━━━━━━━━━━━━━━━ 5s 908us/step - loss: nan - mse: 0.0460 - val_loss: 0.0357 - val_mse: 0.0360
591/591 ━━━━━━━━━━━━━━━━━━━━ 0s 590us/step


0.0 1.0
[[0.03853929 0.01420388]]


In [73]:
#Version 2
lstm_2 = models.Sequential([
    layers.LSTM(32, activation = 'tanh', input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(16, activation = 'tanh'),
    layers.Dropout(0.2),
    layers.Dense(1)
])
lstm_2.compile(optimizer='adam', loss=custom_loss, metrics=['mse'])

historical_data = lstm_2.fit(X_train, y_train, epochs = 50, batch_size=32, validation_data=(X_val, y_val), callbacks=[holdout])
predicted_values = lstm_2.predict(X_val) #Is it wrong to use the validation data in the last line if its also to be used here ?


Epoch 1/50


c:\Users\erkfeldt\AppData\Local\miniconda3\envs\pytorch-gpu\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


5204/5204 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - loss: nan - mse: 0.0460 - val_loss: 0.0357 - val_mse: 0.0360
Epoch 2/50
5204/5204 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - loss: nan - mse: 0.0460 - val_loss: 0.0358 - val_mse: 0.0360
Epoch 3/50
5204/5204 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - loss: nan - mse: 0.0460 - val_loss: 0.0357 - val_mse: 0.0360
591/591 ━━━━━━━━━━━━━━━━━━━━ 1s 768us/step


In [ ]:

print(predicted_values)

[9.56792221e+01 2.53121400e+06]
Epoch 1/50


c:\Users\erkfeldt\AppData\Local\miniconda3\envs\pytorch-gpu\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


5307/5307 ━━━━━━━━━━━━━━━━━━━━ 5s 813us/step - loss: nan
Epoch 2/50
 185/5307 ━━━━━━━━━━━━━━━━━━━━ 4s 821us/step - loss: nan

c:\Users\erkfeldt\AppData\Local\miniconda3\envs\pytorch-gpu\Lib\site-packages\keras\src\callbacks\early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss
  current = self.get_monitor_value(logs)


5307/5307 ━━━━━━━━━━━━━━━━━━━━ 4s 823us/step - loss: nan
Epoch 3/50
5307/5307 ━━━━━━━━━━━━━━━━━━━━ 4s 812us/step - loss: nan
Epoch 4/50
5307/5307 ━━━━━━━━━━━━━━━━━━━━ 4s 806us/step - loss: nan
Epoch 5/50
5307/5307 ━━━━━━━━━━━━━━━━━━━━ 4s 786us/step - loss: nan
Epoch 6/50
5307/5307 ━━━━━━━━━━━━━━━━━━━━ 4s 807us/step - loss: nan
Epoch 7/50
5307/5307 ━━━━━━━━━━━━━━━━━━━━ 4s 839us/step - loss: nan
Epoch 8/50
5307/5307 ━━━━━━━━━━━━━━━━━━━━ 5s 845us/step - loss: nan
Epoch 9/50
5307/5307 ━━━━━━━━━━━━━━━━━━━━ 5s 847us/step - loss: nan
Epoch 10/50
5307/5307 ━━━━━━━━━━━━━━━━━━━━ 4s 843us/step - loss: nan
Epoch 11/50
5307/5307 ━━━━━━━━━━━━━━━━━━━━ 4s 843us/step - loss: nan
Epoch 12/50
5307/5307 ━━━━━━━━━━━━━━━━━━━━ 5s 857us/step - loss: nan
Epoch 13/50
5307/5307 ━━━━━━━━━━━━━━━━━━━━ 4s 838us/step - loss: nan
Epoch 14/50
5307/5307 ━━━━━━━━━━━━━━━━━━━━ 4s 812us/step - loss: nan
Epoch 15/50
5307/5307 ━━━━━━━━━━━━━━━━━━━━ 4s 840us/step - loss: nan
Epoch 16/50
5307/5307 ━━━━━━━━━━━━━━━━━━━━ 4s 828us/s

KeyboardInterrupt: 

Select the network wwith lowest validation error
Run selected network over test set: Whenever LSTM gives value above threshold output 1, else 0.
FOr days with 1, sum profits obtained (p_(t+1)/p_t)-1 where p_t is price o t
Normalise by N x M where N is num stocks and M num of time series
Adjust threshold using val set, but not test set
